In [1]:
import os
import json
from openai import OpenAI

client = OpenAI(
    base_url="https://models.github.ai/inference",
    api_key=os.environ["GITHUB_TOKEN"]
)

In [2]:
# Las herramientas que el agente puede usar
def obtener_tasa_dolar():
    """Simula consultar la TRM actual"""
    return {"tasa": 3710.5, "fecha": "2026-03-15", "fuente": "Banco de la República"}

def calcular_conversion(monto_cop, tasa):
    """Convierte pesos colombianos a dólares"""
    return {"monto_usd": round(monto_cop / tasa, 2), "monto_cop": monto_cop, "tasa_usada": tasa}

In [3]:
# Definición de herramientas para el modelo
herramientas = [
    {
        "type": "function",
        "function": {
            "name": "obtener_tasa_dolar",
            "description": "Obtiene la tasa de cambio actual del dólar en Colombia (TRM)",
            "parameters": {"type": "object", "properties": {}, "required": []}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calcular_conversion",
            "description": "Convierte un monto en pesos colombianos a dólares",
            "parameters": {
                "type": "object",
                "properties": {
                    "monto_cop": {"type": "number", "description": "Monto en pesos colombianos"},
                    "tasa": {"type": "number", "description": "Tasa de cambio a usar"}
                },
                "required": ["monto_cop", "tasa"]
            }
        }
    }
]

In [16]:
def ejecutar_herramienta(nombre, argumentos):
    if nombre == "obtener_tasa_dolar":
        return obtener_tasa_dolar()
    elif nombre == "calcular_conversion":
        return calcular_conversion(**argumentos)

def agente(pregunta):
    mensajes = [{"role": "user", "content": pregunta}]
    
    while True:
        respuesta = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=mensajes,
            tools=herramientas
        )
        
        mensaje = respuesta.choices[0].message
        
        # Si no hay tool calls, el agente terminó
        if not mensaje.tool_calls:
            print(mensajes)
            return mensaje.content
        
        # Ejecutar cada herramienta que pidió el modelo
        mensajes.append(mensaje)
        for tool_call in mensaje.tool_calls:
            nombre = tool_call.function.name
            argumentos = json.loads(tool_call.function.arguments)
            resultado = ejecutar_herramienta(nombre, argumentos)
            
            print(f"  → Herramienta usada: {nombre}({argumentos})")
            
            mensajes.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(resultado)
            })

In [17]:
# Prueba
print("Pregunta: ¿Cuántos dólares son 500,000 pesos?")
respuesta = agente("¿Cuántos dólares son 500,000 pesos colombianos?")
print(f"Respuesta: {respuesta}\n")

Pregunta: ¿Cuántos dólares son 500,000 pesos?
  → Herramienta usada: obtener_tasa_dolar({})
  → Herramienta usada: calcular_conversion({'monto_cop': 500000, 'tasa': 3710.5})
[{'role': 'user', 'content': '¿Cuántos dólares son 500,000 pesos colombianos?'}, ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_KKFnSbKKpEHAV4TUS5Nla3aM', function=Function(arguments='{}', name='obtener_tasa_dolar'), type='function')]), {'role': 'tool', 'tool_call_id': 'call_KKFnSbKKpEHAV4TUS5Nla3aM', 'content': '{"tasa": 3710.5, "fecha": "2026-03-15", "fuente": "Banco de la Rep\\u00fablica"}'}, ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_BBjb2KI5tVhk4qJ7QS3DEtQM', function=Function(arguments='{"monto_cop":500000,"tasa":3710.5}', name='calcular_conversion'), ty